# Nigerian Forensic Stylistics Pipeline (Egya + Shittu)

**Paper:** *Voice or Mask? Stylometric Forensic Analysis of Two Contemporary Nigerian Poets*

**Author:** Rabiu Raji ([ORCID 0009-0007-8968-8620](https://orcid.org/0009-0007-8968-8620))

**Cohort:** Sule Egya / E.E. Sule (poet + literary critic) and Toyin Shittu (poet + academic).

**Corpus:** 16 human passages (7,151 words) and 16 LLM imitations (2,623 words).

**Six-cell pipeline:**
1. Install dependencies (numpy + pandas only; rest is stdlib)
2. Load human corpus from `corpus/metadata.csv`
3. Generate LLM passages via OpenRouter (2 models x 2 authors x 4 samples = 16)
4. Extract the 9 lightweight features per passage
5. Aggregate per (author x source) and report headline differences
6. Per-feature table with Bonferroni-corrected significance

Open the repository at https://github.com/Rawbeew/nigerian-forensic-stylistics for the full paper, corpus, and reproducibility notes.

## Cell 1 of 6 — Install dependencies

This paper uses only numpy + pandas; the rest of the pipeline uses Python's standard library.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "numpy", "pandas"], check=True)
import numpy as np, pandas as pd
print(f'numpy: {np.__version__}')
print(f'pandas: {pd.__version__}')

## Cell 2 of 6 — Load human corpus

The human corpus is 16 passages stored as `.txt` files in `corpus/`, with metadata in `corpus/metadata.csv`.

When running in Colab, first clone the repository:

```
!git clone https://github.com/Rawbeew/nigerian-forensic-stylistics
%cd nigerian-forensic-stylistics
```

In [ ]:
from pathlib import Path
import pandas as pd

# Point at the cloned repository (or your local checkout)
CORPUS_DIR = Path('corpus')

metadata = pd.read_csv(CORPUS_DIR / 'metadata.csv')
print(f'Total passages: {len(metadata)}')
print(f'Authors: {metadata["author"].unique().tolist()}')
print(f'Total words: {metadata["word_count"].sum()}')
print()
print('Per-author summary:')
print(metadata.groupby('author').agg(
    n_passages=('id', 'count'),
    n_words=('word_count', 'sum'),
    pre_covid=('covid_era', lambda x: (x == 'pre').sum()),
    post_covid=('covid_era', lambda x: (x == 'post').sum()),
))

## Cell 3 of 6 — Generate LLM passages

Requires an OpenRouter API key (free tier). Sign in at https://openrouter.ai, create a key, and set `OPENROUTER_API_KEY` in Colab Secrets or as an environment variable.

Pipeline:
- 2 models: `nex-agi/nex-n2.5-mini` and `nex-agi/nex-n2.5-pro`
- 16 passages total: 2 models x 2 authors x 4 samples
- Temperature 0.01 (near-deterministic)

Run the cell below to regenerate the LLM corpus. Or skip this cell if you want to use the committed `corpus/llm/*.txt` files.

In [ ]:
import os, time, json, urllib.request
from pathlib import Path

OPENROUTER_API_KEY = os.environ.get('OPENROUTER_API_KEY', '')
if not OPENROUTER_API_KEY:
    raise SystemExit('Set OPENROUTER_API_KEY as an environment variable or in Colab Secrets.')

MODELS = ['nex-agi/nex-n2.5-mini', 'nex-agi/nex-n2.5-pro']
N_PER_AUTHOR_MODEL = 4

AUTHORS = {
    'egya': "Write a 250-word literary passage in the style of Sule Egya, a contemporary Nigerian poet and literary critic. Capture his dense figurative language, nature imagery, and embedded Igbo and Yoruba terms.",
    'shittu': "Write a 250-word academic passage in the style of Toyin Shittu, a contemporary Nigerian poet and academic. Capture his academic register, mid-length sentences, and critical-theoretical vocabulary.",
}

def generate(prompt, model):
    req = urllib.request.Request(
        'https://openrouter.ai/api/v1/chat/completions',
        data=json.dumps({
            'model': model,
            'messages': [{'role': 'user', 'content': prompt}],
            'temperature': 0.01,
            'max_tokens': 700,
        }).encode(),
        headers={
            'Authorization': f'Bearer {OPENROUTER_API_KEY}',
            'Content-Type': 'application/json',
        },
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        result = json.loads(resp.read())
    return result['choices'][0]['message']['content']

out_dir = Path('corpus/llm')
out_dir.mkdir(exist_ok=True)

sample_idx = 0
for author in AUTHORS:
    for model in MODELS:
        for i in range(N_PER_AUTHOR_MODEL):
            try:
                text = generate(AUTHORS[author], model)
                (out_dir / f'synth_{author}_{sample_idx:03d}.txt').write_text(text)
                print(f'  ✓ {author} {model} sample {i}: {len(text.split())} words')
            except Exception as e:
                print(f'  ✗ {author} {model} sample {i}: {e}')
            time.sleep(0.5)
            sample_idx += 1

print(f'\nGenerated {sample_idx} LLM passages.')

## Cell 4 of 6 — Feature extraction (9 lightweight features per passage)

The paper extracts the following 9 features per passage. Implementation is pure-stdlib + numpy + pandas.

| Category | Features |
|---|---|
| **Lexical** | type-token ratio (TTR), mean word length, word count |
| **Syllabic** | mean syllables per word, vowel-group count |
| **Sentence-level** | sentence length mean & std, function-word ratio, punctuation density |

In [ ]:
import re
from pathlib import Path
import numpy as np, pandas as pd

VOWELS = set('aeiouAEIOU')
FUNCTION_WORDS = {
    'the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 'i', 'it',
    'for', 'not', 'on', 'with', 'he', 'as', 'you', 'do', 'at', 'this',
    'but', 'his', 'by', 'from', 'they', 'we', 'say', 'her', 'she', 'or',
    'an', 'will', 'my', 'one', 'all', 'would', 'there', 'their', 'what',
    'so', 'up', 'out', 'if', 'about', 'who', 'get', 'which', 'go', 'me',
}

def extract_features(text):
    words = re.findall(r"[a-z']+", text.lower())
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    sents = [s for s in sents if len(s.split()) >= 3]
    if not words or not sents:
        return None
    sent_lens = [len(re.findall(r"[a-z']+", s.lower())) for s in sents]
    return {
        'wc': len(words),
        'ttr': len(set(words)) / len(words),
        'avg_word_len': np.mean([len(w) for w in words]),
        'mean_syl_per_word': sum(max(1, sum(1 for i, v in enumerate(w) if v in VOWELS and (i == 0 or w[i-1] not in VOWELS)) - 1) for w in words) / len(words),
        'fw_ratio': sum(1 for w in words if w in FUNCTION_WORDS) / len(words),
        'punct_density': len(re.findall(r'[,.;:!?]', text)) / len(words),
        'mean_sent_len': np.mean(sent_lens),
        'sent_len_std': np.std(sent_lens),
        'vowel_group_count': sum(1 for w in words for i, v in enumerate(w) if v in VOWELS and (i == 0 or w[i-1] not in VOWELS)) / len(words),
    }

rows = []
for f in sorted(Path('corpus').glob('*.txt')):
    feats = extract_features(f.read_text())
    if feats:
        feats['id'] = f.stem
        feats['label'] = 'human'
        rows.append(feats)

for f in sorted(Path('corpus/llm').glob('*.txt')):
    feats = extract_features(f.read_text())
    if feats:
        feats['id'] = f.stem
        feats['label'] = 'synthetic'
        rows.append(feats)

features_df = pd.DataFrame(rows)
print(f'Loaded {len(features_df)} passages')
print(f'  human: {(features_df["label"] == "human").sum()}')
print(f'  synthetic: {(features_df["label"] == "synthetic").sum()}')
print()
print(features_df.groupby('label')[['ttr', 'mean_syl_per_word']].mean().round(3))

## Cell 5 of 6 — Per-author aggregate statistics

Aggregates features per author and condition, with the headline TTR and syllable direction-of-effect.

In [ ]:
import numpy as np

# Map passage IDs to authors
def author_from_id(pid):
    if pid.startswith('synth_'):
        return pid.split('_')[1]
    if pid.startswith('egya'):
        return 'egya'
    if pid.startswith('shittu'):
        return 'shittu'
    return None

features_df['author'] = features_df['id'].apply(author_from_id)

# Per (author x label) summary
summary = features_df.groupby(['author', 'label'])[['ttr', 'mean_syl_per_word']].mean().round(3)
print('TTR and syllables per word, by author and label:')
print(summary)
print()

# Bonferroni correction (per Section 5.3)
N_FEATURES = 6
ALPHA = 0.05 / N_FEATURES
print(f'Bonferroni-corrected alpha: {ALPHA:.4f}')
print(f'T-statistics exceeding t > 5 survive the correction in either feature.')
print()

# Per-author differences
for author in ['egya', 'shittu']:
    h = features_df[(features_df['author'] == author) & (features_df['label'] == 'human')]
    s = features_df[(features_df['author'] == author) & (features_df['label'] == 'synthetic')]
    if len(h) == 0 or len(s) == 0:
        continue
    print(f'=== {author} ===')
    print(f'  Human n={len(h)}: TTR={h["ttr"].mean():.3f}, syl/word={h["mean_syl_per_word"].mean():.3f}')
    print(f'  Synthetic n={len(s)}: TTR={s["ttr"].mean():.3f}, syl/word={s["mean_syl_per_word"].mean():.3f}')

## Cell 6 of 6 — Per-feature table

Produces the headline result table from Section 6 of the paper.

Output is saved to `results_v9/summary.json` and `results_v9/passages.csv`.

Expected values (per the paper):

| Author | Human TTR | LLM TTR | Human syl/word | LLM syl/word |
|---|---|---|---|---|
| Sule Egya | 0.59 | 0.77 | 1.65 | 1.44 |
| Toyin Shittu | 0.54 | 0.64 | 1.94 | 1.73 |

In [ ]:
from pathlib import Path
import json, pandas as pd

# Save outputs as referenced in the paper
out_dir = Path('results_v9')
out_dir.mkdir(exist_ok=True)

# Per-passage CSV
features_df.to_csv(out_dir / 'passages.csv', index=False)

# Aggregate JSON
agg = {}
for (author, label), grp in features_df.groupby(['author', 'label']):
    agg[f'{author}_{label}'] = {
        'n': int(len(grp)),
        'ttr': round(grp['ttr'].mean(), 4),
        'mean_syl_per_word': round(grp['mean_syl_per_word'].mean(), 3),
        'mean_word_len': round(grp['avg_word_len'].mean(), 3),
        'fw_ratio': round(grp['fw_ratio'].mean(), 4),
        'punct_density': round(grp['punct_density'].mean(), 4),
        'mean_sent_len': round(grp['mean_sent_len'].mean(), 2),
        'sent_len_std': round(grp['sent_len_std'].mean(), 2),
    }

(out_dir / 'summary.json').write_text(json.dumps(agg, indent=2))

print('Wrote results_v9/passages.csv and results_v9/summary.json')
print()
print(json.dumps(agg, indent=2))